# Company Lead Pipeline

This notebook combines three stages:

1. **Clean companies** — extract company names from emails, remove blank names and duplicates, extract city.
2. **Website checker** — guess and verify each company's own-domain website; classify site condition and company type.
3. **Lead qualification** — classify each company as a Custom Build Candidate, PulseBoard Candidate, or Low Fit / Unqualified.

**Input:** one Excel file with company name, phone, email, state, and full address.  
**Outputs:** `cleaned_companies.xlsx`, `website_check_results.xlsx`, and `lead_qualification_results.xlsx`.

Run the cells in order. The website-check stage uses domain guessing only (no web search), so it is faster and avoids search rate limits.

In [1]:
%pip -q install wordsegment beautifulsoup4 lxml openpyxl tqdm requests ddgs

Note: you may need to restart the kernel to use updated packages.


## 1. Configuration
Set your input Excel path and output folder. Column names are auto-detected but can be overridden below.

In [2]:
INPUT_FILE = "C:/Users/Ratna/Desktop/Think Scaler/Leadsdata/leads_data.xlsx"
OUTPUT_DIR = "C:/Users/Ratna/Downloads"

CLEANED_FILE = f"{OUTPUT_DIR}/cleaned_companies.xlsx"
WEBSITE_FILE = f"{OUTPUT_DIR}/website_check_results.xlsx"
QUALIFIED_FILE = f"{OUTPUT_DIR}/lead_qualification_results.xlsx"

# Set these manually only if auto-detection chooses the wrong columns.
FORCE_COLUMNS = {
    # "company": "Business Name",
    # "email": "Registered Email",
    # "address": "Address",
    # "state": "State",
}

MAX_ROWS = None          # e.g. 100 for testing; None = all rows
WORKERS = 16             # parallel website checks
TIMEOUT = 8
SAVE_EVERY = 1000
STEP_A_LIMIT = 120       # hard seconds per company for website checking
CHECKPOINT_EVERY = 25


## 2. Clean companies
Extracts a company name from the email local part when the business-name field is blank, removes rows without a company name, removes duplicates, and extracts a city from the address.

In [3]:
import re, difflib
import pandas as pd
from wordsegment import load as ws_load, segment
ws_load()

df = pd.read_excel(INPUT_FILE, dtype=str)
if MAX_ROWS:
    df = df.head(MAX_ROWS).copy()

print("Input columns:", df.columns.tolist())
print("Input rows:", len(df))

# Remove completely empty rows and exact duplicate rows
df = df.dropna(how="all").drop_duplicates().reset_index(drop=True)

def find_col(keywords):
    for key in keywords:
        for col in df.columns:
            if key in str(col).lower():
                return col
    return None

EMAIL_COL = FORCE_COLUMNS.get("email") or find_col(["mail"])
ADDRESS_COL = FORCE_COLUMNS.get("address") or find_col(["address"])
STATE_COL = FORCE_COLUMNS.get("state") or find_col(["state"])
COMPANY_COL = FORCE_COLUMNS.get("company") or find_col(["company", "business", "firm", "name"])

if COMPANY_COL is None or COMPANY_COL == EMAIL_COL:
    COMPANY_COL = "Company Name"
    df[COMPANY_COL] = ""

print("Company:", COMPANY_COL, "| Email:", EMAIL_COL, "| Address:", ADDRESS_COL, "| State:", STATE_COL)
assert EMAIL_COL, "Could not find an email column."
assert ADDRESS_COL, "Could not find an address column."


Input columns: ['Business Name', 'Registered Mobile Number', 'Registered Email ', 'State', 'Address']
Input rows: 6152
Company: Business Name | Email: Registered Email  | Address: Address | State: State


In [4]:
BUSINESS_WORDS = {
    "pvt","ltd","private","limited","llp","inc","corp","co","company","group",
    "enterprise","enterprises","industries","industry","industrial","traders","trading",
    "trade","exports","export","imports","import","metals","metal","steel","steels",
    "construction","constructions","builders","builder","infra","infrastructure",
    "engineering","engineers","engg","electricals","electrical","electronics",
    "technologies","technology","tech","solutions","services","systems","software",
    "pharma","pharmaceuticals","chemicals","chemical","plastics","polymers","textiles",
    "textile","garments","fabrics","foods","food","agro","agri","mining","minerals",
    "logistics","transport","transports","travels","motors","auto","automobiles",
    "machinery","machines","tools","hardware","power","energy","solar","mart",
    "stores","store","agency","agencies","associates","consultants","consultancy",
    "marketing","sales","distributors","suppliers","supplies","packaging","paper",
    "cement","ceramics","granite","marble","furniture","jewellers","jewellery",
    "developers","realty","estate","properties","hospital","clinic","labs","laboratories",
    "works","workshop","fabricators","fabrication","contractors","contractor","projects",
    "ventures","holdings","international","global","overseas","impex","udyog",
    "bharat","india","indian","national","mfg","manufacturing","manufacturers"
}

GENERIC_PREFIXES = {
    "info","admin","contact","sales","support","accounts","account","hr","mail",
    "office","enquiry","inquiry","hello","service","billing","noreply","test","user"
}

def norm(text):
    return re.sub(r"[^a-z]", "", str(text).lower())

def is_blank(value):
    return pd.isna(value) or str(value).strip().lower() in {
        "", "nan", "none", "null", "-", "na", "n/a"
    }

def extract_company_from_email(email, original_company=None):
    if not isinstance(email, str) or "@" not in email:
        return ""

    local_part = email.split("@")[0].lower()
    clean = re.sub(r"[^a-z]", "", local_part)

    if len(clean) < 4 or clean in GENERIC_PREFIXES:
        return ""

    # If an existing company name is similar, accept the email-derived name.
    if original_company and not is_blank(original_company):
        original_norm = norm(original_company)
        if original_norm and difflib.SequenceMatcher(None, clean, original_norm).ratio() >= 0.6:
            return clean

    # Accept only if it looks business-like.
    words = segment(clean)
    if any(word in BUSINESS_WORDS for word in words):
        return clean

    return ""

# Fill only blank company names from email.
blank_mask = df[COMPANY_COL].apply(is_blank)
df.loc[blank_mask, COMPANY_COL] = df.loc[blank_mask, EMAIL_COL].apply(
    lambda email: extract_company_from_email(email)
)
df[COMPANY_COL] = df[COMPANY_COL].fillna("").astype(str).str.strip()

filled_count = (blank_mask & (df[COMPANY_COL] != "")).sum()
print(f"Existing company names: {(~blank_mask).sum()}")
print(f"Filled from email: {filled_count}")
print(f"Still blank: {(df[COMPANY_COL] == '').sum()}")

# Remove rows without a company name.
before = len(df)
df = df[df[COMPANY_COL] != ""].reset_index(drop=True)
print(f"Removed {before - len(df)} rows without a company name; {len(df)} remain.")


Existing company names: 4740
Filled from email: 214
Still blank: 1185
Removed 1185 rows without a company name; 4954 remain.


In [5]:
CITIES = [
    "Mumbai","Delhi","New Delhi","Bengaluru","Bangalore","Hyderabad","Ahmedabad","Chennai",
    "Kolkata","Pune","Jaipur","Surat","Lucknow","Kanpur","Nagpur","Indore","Thane",
    "Bhopal","Visakhapatnam","Patna","Vadodara","Ghaziabad","Ludhiana","Agra","Nashik",
    "Faridabad","Meerut","Rajkot","Varanasi","Srinagar","Aurangabad","Dhanbad","Amritsar",
    "Navi Mumbai","Allahabad","Prayagraj","Ranchi","Howrah","Coimbatore","Jabalpur",
    "Gwalior","Vijayawada","Jodhpur","Madurai","Raipur","Kota","Guwahati","Chandigarh",
    "Solapur","Hubli","Mysore","Mysuru","Tiruchirappalli","Bareilly","Aligarh","Moradabad",
    "Jalandhar","Bhubaneswar","Cuttack","Rourkela","Sambalpur","Berhampur","Puri","Balasore",
    "Jamshedpur","Bokaro","Noida","Gurgaon","Gurugram","Greater Noida","Dehradun","Haridwar",
    "Kochi","Thiruvananthapuram","Kozhikode","Thrissur","Mangalore","Belgaum","Salem",
    "Vellore","Tiruppur","Erode","Bhilai","Durg","Bilaspur","Udaipur","Ajmer","Bikaner",
    "Siliguri","Durgapur","Asansol","Jamnagar","Bhavnagar","Gandhinagar","Vapi","Ankleshwar"
]

CITIES = sorted(set(CITIES), key=len, reverse=True)
CITY_PATTERNS = [(city, re.compile(r"\b" + re.escape(city.lower()) + r"\b")) for city in CITIES]

def extract_city(address, state=None):
    if not isinstance(address, str) or not address.strip():
        return ""

    text = address.lower()

    for city, pattern in CITY_PATTERNS:
        if pattern.search(text):
            return city

    state_text = str(state).strip().lower() if state else ""
    parts = [part.strip() for part in re.split(r"[,\n;]", text) if part.strip()]
    cleaned_parts = []

    for part in parts:
        part = re.sub(r"\b\d{6}\b", "", part)
        part = re.sub(r"[^a-z\s]", " ", part)
        part = re.sub(r"\s+", " ", part).strip()
        if part and part != state_text and part not in {"india", "in"}:
            cleaned_parts.append(part)

    return cleaned_parts[-1].title() if cleaned_parts else ""

df["City Name"] = [
    extract_city(address, df[STATE_COL].iloc[index] if STATE_COL else None)
    for index, address in enumerate(df[ADDRESS_COL])
]

# Remove duplicate companies based on normalized name + phone + email + address.
df["_dedupe_key"] = (
    df[COMPANY_COL].str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
    + "|" + df[EMAIL_COL].fillna("").str.lower()
    + "|" + df[ADDRESS_COL].fillna("").str.lower()
)
before = len(df)
df = df.drop_duplicates(subset="_dedupe_key").drop(columns="_dedupe_key").reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate rows; {len(df)} remain.")

df.to_excel(CLEANED_FILE, index=False)
print(f"Saved cleaned companies: {CLEANED_FILE}")
df.head()


Removed 1 duplicate rows; 4953 remain.
Saved cleaned companies: C:/Users/Ratna/Downloads/cleaned_companies.xlsx


,Business Name,Registered Mobile Number,Registered Email,State,Address,City Name
0,SANDHYA JEWELLERS,'9893280101,casanjaymahajan1006@gmail.com,Madhya Pradesh,"SHOP NO 222, 25, ZAWAR COMPLEX, Sarafa Street,...",Indore
1,arshmetalsprivatelimited,'9182856304,arshmetalsprivatelimited@gmail.com,NaN,NaN,
2,vjaindustries,'8825420560,vjaindustriesklk@gmail.com,NaN,NaN,
3,INDIAN PLASTER OF PARIS,'9792148886,shamshadalimanihar306@gmail.com,Andhra Pradesh,"9-11-5, REVENUE WARD NO 9, AMBATI VARI STREET...",East Godavari
4,PRABHU CATERERS,'9901906565,sn2104@gmail.com,Karnataka,"1ST FLOOR, 37/1, KHAZEE STREET CROSS M D BLO...",Bengaluru


## 3. Website checker
Guesses likely own-domain websites from each company name, opens the homepage, verifies the company name appears on it, assesses site condition, and classifies company type. No web search is performed.

In [6]:
import re, time, random, difflib, warnings, socket, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import urlparse
from datetime import datetime

import requests, urllib3
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from wordsegment import load as _ws_load, segment
try:
    from ddgs import DDGS
except ImportError:
    from duckduckgo_search import DDGS

_ws_load()
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.filterwarnings("ignore")

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ---------------- SETTINGS ----------------
MAX_ROWS     = None          # e.g. 50 to test on the first 50 rows, None = whole file
WORKERS      = 32            # parallel threads for Step A (domain guessing). Try 16-64
SEARCH_REMAINING = True      # False = skip web search entirely; companies whose site was not found
                             # by Step A are marked "Not found (search skipped)"  -> fastest
SEARCH_WORKERS = 4           # parallel threads for Step B (web search)
MIN_SEARCH_INTERVAL = 0.6    # seconds between searches across ALL threads (raised automatically if rate-limited)
SEARCH_QUERIES = 1           # 1 = one search per company (fast), 2 = retry with a second query
SEARCH_BACKEND = "ddgs"      # "ddgs" = free, but rate-limited | "serper" = paid API (serper.dev), much faster
SERPER_API_KEY = ""          # only needed when SEARCH_BACKEND = "serper"
TIMEOUT      = 8             # seconds to wait for a website
MAX_RESULTS  = 8             # search results looked at per query
MAX_CANDIDATES = 2           # websites opened per query
SAVE_EVERY   = 1000          # saves a partial Excel after every N companies
THIS_YEAR    = datetime.now().year
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                         "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
           "Accept-Language": "en-US,en;q=0.9"}

In [7]:
LEGAL_WORDS = {"pvt", "ltd", "private", "limited", "llp", "inc", "corp", "corporation", "co",
               "company", "the", "and", "of", "opc", "pte", "llc", "m", "s"}
GENERIC_WORDS = {"india", "indian", "bharat", "enterprises", "enterprise", "industries", "industry",
                 "traders", "trading", "solutions", "services", "systems", "group", "international",
                 "global", "national", "agency", "associates", "sales", "works", "projects",
                 "metals", "metal", "steel", "steels", "construction", "constructions",
                 "engineering", "engineers", "technologies", "technology", "infra", "exports",
                 "export", "imports", "import", "impex", "udyog", "agro", "agri", "foods", "food"}

# Websites that list companies but are NOT the company's own site
DIRECTORY_DOMAINS = {
    "indiamart.com", "justdial.com", "justdial.in", "tradeindia.com", "exportersindia.com",
    "sulekha.com", "facebook.com", "instagram.com", "linkedin.com", "twitter.com", "x.com",
    "youtube.com", "pinterest.com", "quora.com", "zaubacorp.com", "tofler.in", "thecompanycheck.com",
    "indiafilings.com", "instafinancials.com", "companydetails.in", "mca.gov.in", "glassdoor.com",
    "glassdoor.co.in", "ambitionbox.com", "naukri.com", "indeed.com", "wikipedia.org",
    "yellowpages.com", "yellowpages.in", "yellowpages.co.in", "mapquest.com", "google.com",
    "bing.com", "yelp.com", "alibaba.com", "made-in-china.com", "amazon.in", "amazon.com",
    "flipkart.com", "crunchbase.com", "dnb.com", "zoominfo.com", "bloomberg.com",
    "opencorporates.com", "cleartax.in", "corporatedir.com", "companieslist.in", "kompass.com",
    "globalsources.com", "europages.com", "clutch.co", "apollo.io", "rocketreach.co",
    "cbinsights.com", "owler.com", "tracxn.com", "mouthshut.com", "bizapedia.com", "dandb.com",
    "trustpilot.com", "medium.com", "mapsofindia.com", "asklaila.com", "connect2india.com",
    "fundoodata.com", "companiesindia.in", "biz-india.com", "ebay.com", "olx.in", "quikr.com",
    "tripadvisor.com", "tripadvisor.in", "zomato.com", "swiggy.com", "mca.gov.in", "gst.gov.in",
    "vymo.com", "grotal.com", "indiabizclub.com", "indianyellowpages.com", "business-standard.com",
    "economictimes.com", "livemint.com", "moneycontrol.com", "bing.com", "duckduckgo.com",
}
# Free site builders: the company does have a site, but not on its own domain
FREE_HOST_SUFFIXES = ("wixsite.com", "blogspot.com", "wordpress.com", "weebly.com", "webflow.io",
                      "github.io", "netlify.app", "vercel.app", "myshopify.com", "sites.google.com")

def name_tokens(name):
    words = re.findall(r"[a-z0-9]+", str(name).lower())
    out = []
    for w in words:
        if w in LEGAL_WORDS:
            continue
        # names that came from emails are glued together: shmetalsprivatelimited
        if len(w) > 10 and w.isalpha():
            out.extend(x for x in segment(w) if x not in LEGAL_WORDS)
        else:
            out.append(w)
    return out

def significant(tokens):
    s = [t for t in tokens if t not in GENERIC_WORDS and len(t) >= 2]
    return s or [t for t in tokens if len(t) >= 2] or tokens

def coverage(words, text):
    """share of words that appear in text"""
    if not words:
        return 0.0
    hit = 0
    for w in words:
        pat = r"\b" + re.escape(w) + (r"\b" if len(w) < 4 else "")
        if re.search(pat, text):
            hit += 1
    return hit / len(words)

def reg_domain(host):
    host = host.lower().split(":")[0]
    if host.startswith("www."):
        host = host[4:]
    parts = host.split(".")
    if len(parts) <= 2:
        return host
    if len(parts[-1]) == 2 and parts[-2] in {"co", "com", "org", "net", "gov", "ac", "edu", "nic", "res"}:
        return ".".join(parts[-3:])
    return ".".join(parts[-2:])

def is_free_host(host):
    return host.endswith(FREE_HOST_SUFFIXES)

def is_directory(host):
    return (not is_free_host(host)) and reg_domain(host) in DIRECTORY_DOMAINS

def site_label(host):
    host = host.lower().split(":")[0]
    if host.startswith("www."):
        host = host[4:]
    if is_free_host(host):
        return host.split(".")[0]
    return reg_domain(host).split(".")[0]

def domain_strength(lab, sig, joined):
    """2 = domain clearly contains the company name, 1 = probably, 0 = no"""
    lab = re.sub(r"[^a-z0-9]", "", lab)
    if len(lab) < 3 or not joined:
        return 0
    if len(lab) >= 4 and len(joined) >= 4 and joined in lab:
        return 2
    if len(lab) >= 4 and lab in joined and len(lab) >= 0.75 * len(joined):
        return 2
    long_sig = [t for t in sig if len(t) >= 3]
    if long_sig and all(t in lab for t in long_sig):
        return 1
    if difflib.SequenceMatcher(None, lab, joined).ratio() >= 0.8:
        return 1
    return 0

def short_loc(address):
    if not isinstance(address, str):
        return ""
    a = re.sub(r"\b\d{6}\b", " ", address)
    parts = [p.strip() for p in re.split(r"[,\n;]", a) if p.strip()]
    return ", ".join(parts[-2:])

In [8]:
_search_lock = threading.Lock()
_last_search = [0.0]
_interval = [MIN_SEARCH_INTERVAL]

def _wait_turn():
    """spaces searches out across all threads so the search engine does not block us"""
    with _search_lock:
        wait = _last_search[0] + _interval[0] - time.time()
        if wait > 0:
            time.sleep(wait)
        _last_search[0] = time.time()

def web_search(query):
    """returns (results, search_worked)"""
    for attempt in range(3):
        _wait_turn()
        try:
            if SEARCH_BACKEND == "serper":
                r = requests.post("https://google.serper.dev/search",
                                  headers={"X-API-KEY": SERPER_API_KEY, "Content-Type": "application/json"},
                                  json={"q": query, "gl": "in", "num": MAX_RESULTS}, timeout=15)
                r.raise_for_status()
                res = [{"title": o.get("title", ""), "href": o.get("link", ""), "body": o.get("snippet", "")}
                       for o in r.json().get("organic", [])]
            else:
                res = list(DDGS().text(query, region="in-en", max_results=MAX_RESULTS))
            _interval[0] = max(MIN_SEARCH_INTERVAL, _interval[0] * 0.95)    # speed up again after success
            return res, True
        except Exception as e:
            if "no results" in str(e).lower():
                return [], True
            _interval[0] = min(8.0, _interval[0] * 1.5)                     # back off when rate-limited
            time.sleep(3 * (attempt + 1))
    return [], False

def fetch_home(url):
    p = urlparse(url)
    first = p.scheme or "https"
    out = {"reachable": False, "status": None, "final_url": "", "html": "",
           "load_s": None, "ssl_error": False, "error": ""}
    for scheme in ([first] + (["http"] if first == "https" else [])):
        target = f"{scheme}://{p.netloc}/"
        t0 = time.time()
        for verify in (True, False):
            try:
                r = requests.get(target, headers=HEADERS, timeout=(5, TIMEOUT),
                                 allow_redirects=True, verify=verify)
                ct = r.headers.get("content-type", "").lower()
                out.update(reachable=True, status=r.status_code, final_url=r.url,
                           html=r.text[:400000] if ("html" in ct or not ct) else "",
                           load_s=time.time() - t0, ssl_error=(not verify))
                return out
            except requests.exceptions.SSLError:
                continue
            except Exception as e:
                out["error"] = type(e).__name__
                break
    return out

PARKED = (r"under construction|coming soon|domain (?:is )?for sale|buy this domain|may be for sale|"
          r"parked|website expired|account suspended|default web page|index of /|it works!|"
          r"apache2 (?:ubuntu )?default|site not found|future home of|under maintenance")
OLD_TECH = r"<frameset|<marquee|<blink|<font[ >]|\.swf|shockwave-flash|x-shockwave"
MODERN = (r"bootstrap|tailwind|elementor|wixstatic|shopify|squarespace|webflow|_next/|react-dom|"
          r"vue(?:\.min)?\.js|swiper|fontawesome|font-awesome|googletagmanager|gtag\(")

def page_features(html):
    low = html.lower()
    soup = BeautifulSoup(html, "lxml")
    title = soup.title.string.strip() if soup.title and soup.title.string else ""
    d = soup.find("meta", attrs={"name": re.compile("^description$", re.I)})
    desc = (d.get("content") or "").strip() if d else ""
    heads = " ".join(h.get_text(" ", strip=True) for h in soup.find_all(["h1", "h2"])[:10])
    n_links = len(soup.find_all("a", href=True))
    for t in soup(["script", "style", "noscript"]):
        t.decompose()
    text = re.sub(r"\s+", " ", soup.get_text(" ")).strip()
    years = []
    for m in re.finditer(r"(?:©|&copy;|copyright)\D{0,40}((?:19|20)\d{2})(?:\s*(?:-|–|—|to)\s*((?:19|20)\d{2}))?",
                         text.lower()):
        years += [int(g) for g in m.groups() if g]
    years = [y for y in years if y <= THIS_YEAR + 1]
    has_meta = bool(desc) or 'property="og:' in low or "application/ld+json" in low
    return {"title": title, "desc": desc, "heads": heads, "text": text, "n_links": n_links,
            "copyright_year": max(years) if years else None, "has_meta": has_meta}

def assess_site(page, free_host=False):
    """returns (condition, score, notes, features)"""
    if not page["reachable"]:
        return "Very poor", -9, ["Website not loading (" + (page["error"] or "no response") + ")"], {}
    st = page["status"]
    if st in (401, 403, 406, 429):
        return "Could not assess", None, [f"Site blocks automated access (HTTP {st})"], {}
    if st >= 400:
        return "Very poor", -9, [f"Website returns HTTP {st}"], {}

    f = page_features(page["html"])
    low = page["html"].lower()
    score, notes = 0, []

    def add(points, why):
        nonlocal score
        score += points
        notes.append(f"{points:+d} {why}")

    if page["ssl_error"]:
        add(-3, "invalid/expired SSL certificate")
    elif page["final_url"].startswith("https"):
        add(2, "uses HTTPS")
    else:
        add(-1, "no HTTPS")
    if re.search(r"<meta[^>]+name=[\"']viewport[\"']", low):
        add(2, "mobile-friendly (viewport tag)")
    else:
        add(-2, "not mobile-friendly")
    if f["has_meta"]:
        add(1, "has SEO/social meta tags")
    if re.search(MODERN, low):
        add(1, "modern framework/CSS")
    tl = len(f["text"])
    if tl >= 1500:
        add(1, "good amount of content")
    elif tl < 300:
        add(-2, "almost no content")
    if f["n_links"] >= 8:
        add(1, "proper navigation")
    if "mailto:" in low or "tel:" in low or "contact" in f["text"].lower():
        add(1, "contact details present")
    if page["load_s"] is not None:
        if page["load_s"] < 3:
            add(1, "loads fast")
        elif page["load_s"] > 6:
            add(-1, "slow to load")
    y = f["copyright_year"]
    if y:
        if y >= THIS_YEAR - 1:
            add(1, f"copyright {y} (kept up to date)")
        elif y <= THIS_YEAR - 4:
            add(-2, f"outdated copyright {y}")
    if re.search(OLD_TECH, low):
        add(-2, "old web technology (frames/flash/marquee)")
    if tl < 1500 and (re.search(PARKED, f["title"].lower()) or re.search(PARKED, f["text"].lower())):
        add(-4, "parked / under construction page")
    if free_host:
        add(-1, "free subdomain, no own domain")

    label = "New / Modern" if score >= 7 else "Requires work" if score >= 3 else "Very poor"
    return label, score, notes, f

In [9]:
CATEGORIES = {
    "E-commerce / Online Retail": ["ecommerce", "e-commerce", "online store", "online shop", "add to cart",
        "shopping cart", "buy now", "marketplace", "shopify", "woocommerce", "checkout", "free shipping"],
    "Agriculture": ["agro", "agri", "agriculture", "agricultural", "farm", "farming", "farmers", "crop", "crops",
        "seeds", "fertilizer", "fertilizers", "pesticide", "irrigation", "tractor", "horticulture",
        "dairy", "poultry", "fisheries", "organic", "nursery", "cattle feed", "rice mill"],
    "Food & Beverage": ["foods", "food products", "restaurant", "bakery", "snacks", "beverages", "catering",
        "spices", "masala", "sweets", "juice", "tea", "coffee", "edible oil", "flour", "frozen food", "namkeen"],
    "Textile & Apparel": ["textile", "textiles", "garments", "apparel", "fabric", "fabrics", "cotton", "yarn",
        "weaving", "knitwear", "saree", "sarees", "clothing", "fashion", "denim", "handloom", "hosiery"],
    "Construction & Real Estate": ["construction", "constructions", "builders", "infra", "infrastructure",
        "real estate", "realty", "developers", "contractors", "civil", "housing", "architects", "interiors",
        "cement", "bricks", "roads", "township", "properties"],
    "Metals, Steel & Mining": ["steel", "steels", "metals", "metal", "iron", "alloy", "alloys", "foundry",
        "casting", "castings", "forging", "mining", "minerals", "aluminium", "aluminum", "copper", "scrap",
        "ferro", "sponge iron", "tmt", "ingots", "coal"],
    "Manufacturing & Engineering": ["engineering", "engineers", "manufacturing", "manufacturers", "machinery",
        "machines", "fabrication", "industries", "industrial", "tools", "equipment", "pumps", "valves",
        "automation", "cnc", "precision", "components", "spare parts", "bearings", "boilers", "fasteners"],
    "Automotive": ["automobile", "automobiles", "automotive", "auto parts", "motors", "vehicle", "vehicles",
        "tyres", "tires", "two wheeler", "garage", "car", "cars", "trucks", "ev charging", "dealership"],
    "IT & Software": ["software", "it services", "technologies", "web development", "app development",
        "digital", "cloud", "saas", "cyber", "data", "consulting", "erp", "website design", "it solutions",
        "artificial intelligence", "mobile apps"],
    "Pharma & Healthcare": ["pharma", "pharmaceutical", "pharmaceuticals", "medicine", "medicines", "drugs",
        "hospital", "clinic", "healthcare", "diagnostics", "medical", "surgical", "ayurvedic", "wellness",
        "laboratories", "labs", "nursing", "dental"],
    "Chemicals & Plastics": ["chemicals", "chemical", "plastics", "plastic", "polymers", "polymer", "resin",
        "paints", "adhesives", "dyes", "solvents", "rubber", "fibre", "pvc"],
    "Logistics & Transport": ["logistics", "transport", "transports", "transportation", "cargo", "courier",
        "freight", "shipping", "warehousing", "packers", "movers", "carriers", "supply chain"],
    "Education & Training": ["school", "college", "institute", "academy", "education", "coaching", "training",
        "university", "learning", "tutorials", "courses", "students"],
    "Energy & Power": ["solar", "power", "energy", "renewable", "wind", "battery", "batteries", "transformer",
        "transformers", "generator", "generators", "biogas", "petroleum", "fuel", "oil and gas"],
    "Electronics & Electricals": ["electronics", "electrical", "electricals", "led", "cables", "wires",
        "switchgear", "circuits", "semiconductor", "appliances", "inverter", "sensors"],
    "Trading & Distribution": ["traders", "trading", "distributors", "distributor", "suppliers", "supplier",
        "wholesale", "wholesalers", "exporters", "exporter", "importers", "importer", "impex", "agency",
        "agencies", "commission agent"],
    "Retail & Consumer Goods": ["store", "stores", "mart", "retail", "supermarket", "showroom", "furniture",
        "jewellery", "jewellers", "jewelry", "gifts", "home decor", "footwear", "toys", "cosmetics"],
    "Hospitality & Travel": ["hotel", "hotels", "resort", "resorts", "travels", "tours", "tourism", "lodge",
        "holidays", "travel agency", "homestay", "banquet"],
    "Financial Services": ["finance", "financial", "capital", "loans", "insurance", "investment",
        "investments", "bank", "nbfc", "microfinance", "securities", "leasing", "chit fund", "wealth"],
    "Media, Printing & Packaging": ["media", "advertising", "printing", "printers", "publications", "studio",
        "films", "packaging", "paper", "cartons", "labels", "corrugated", "offset"],
}
SUBSTRING_STOP = {"auto", "tech", "food", "labs", "tour", "soft", "data", "cars", "shop", "mart", "bank",
                  "fund", "care", "home", "wire", "tool", "farm", "seed", "iron", "coal", "wind", "fuel",
                  "tea", "led", "car", "tmt", "pvc", "erp", "cnc"}

def classify(name, texts, html_low=""):
    name_norm = re.sub(r"[^a-z0-9]+", " ", str(name).lower()).strip()
    name_joined = name_norm.replace(" ", "")
    blob = " ".join(texts).lower()
    scores = {c: 0 for c in CATEGORIES}
    for cat, kws in CATEGORIES.items():
        for kw in kws:
            k = kw.lower()
            kj = k.replace(" ", "")
            pat_name = r"\b" + re.escape(k) + r"\b"
            if re.search(pat_name, name_norm) or (len(kj) >= 4 and kj not in SUBSTRING_STOP and kj in name_joined):
                scores[cat] += 3
            pat_blob = r"\b" + re.escape(k) + (r"\b" if len(k) <= 4 else "")
            if re.search(pat_blob, blob):
                scores[cat] += 1
    if re.search(r"woocommerce|shopify|add[-_ ]to[-_ ]cart|/cart\b|/checkout\b", html_low):
        scores["E-commerce / Online Retail"] += 4
    best = max(scores, key=scores.get)
    return best if scores[best] >= 2 else "Other / Unclassified"

In [10]:
TLDS = [".com", ".in", ".co.in", ".net", ".org"]

def guess_domains(tokens, sig):
    """returns [(domain, needs_location_match)]"""
    out = []
    base = "".join(tokens)
    if 4 <= len(base) <= 40:
        out += [(base + t, False) for t in TLDS]
        if len(tokens) > 1:
            dashed = "-".join(tokens)
            out += [(dashed + ".com", False), (dashed + ".in", False)]
    short = "".join(sig)           # name without generic words: 'Kalpa Tech Solutions' -> kalpatech
    if short != base and 4 <= len(short) <= 40:
        # a one-word name like 'sunrise' is shared by many companies, so the page must also mention the location
        out += [(short + t, len(sig) == 1) for t in TLDS[:3]]
    return out

def dns_host(dom):
    """returns the host name if the domain exists, else None (very fast for non-existent domains)"""
    for h in (dom, "www." + dom):
        try:
            socket.getaddrinfo(h, 80)
            return h
        except OSError:
            continue
    return None

def evaluate_candidate(url, host, strength, sig, loc_words, guessed=False, need_loc=False):
    """opens the homepage and decides whether it is really this company's own site"""
    page = fetch_home(url)
    st = page["status"] or 0
    no_content = (not page["reachable"]) or st in (401, 403, 406, 429) or st >= 400
    cov, loc_hit, parked = 0.0, False, False
    if not no_content:
        f = page_features(page["html"])
        ptxt = " ".join([f["title"], f["desc"], f["heads"], f["text"][:3000]]).lower()
        cov = coverage(sig, ptxt)
        loc_hit = any(w in ptxt for w in loc_words)
        parked = len(f["text"]) < 1500 and bool(re.search(PARKED, f["title"].lower() + " " + f["text"].lower()))
    if guessed:
        # a guessed domain may be parked, for sale or belong to someone else: the name must be on the page
        need = 0.99 if len(sig) <= 2 else 0.66
        if no_content or parked or cov < need or (need_loc and not loc_hit):
            return None
        conf = "High" if loc_hit else "Medium"
    else:
        if no_content:
            accept = strength >= 1
        else:
            accept = strength == 2 or (strength == 1 and cov >= 0.5) or (strength == 0 and cov >= 0.99 and len(sig) >= 2)
        if not accept:
            return None
        if no_content:
            conf = "Medium" if strength == 2 else "Low"
        elif strength >= 1 and cov >= 0.5 and loc_hit:
            conf = "High"
        else:
            conf = "Medium" if (strength >= 1 or cov >= 0.99) else "Low"
    return dict(url=page["final_url"] or url, page=page, host=host, conf=conf)

def guess_website(name, address):
    """Step A: no search needed"""
    tokens = name_tokens(name)
    sig = significant(tokens)
    loc_words = re.findall(r"[a-z]{4,}", short_loc(address).lower())
    for dom, need_loc in guess_domains(tokens, sig):
        host = dns_host(dom)
        if not host:
            continue
        found = evaluate_candidate("https://" + host + "/", host, 2, sig, loc_words, guessed=True, need_loc=need_loc)
        if found:
            return found
    return None

def search_website(name, address):
    """Step B: web search. returns (found, snippets, search_worked)"""
    tokens = name_tokens(name)
    sig = significant(tokens)
    joined = "".join(tokens)
    loc = short_loc(address)
    loc_words = re.findall(r"[a-z]{4,}", loc.lower())
    snippets, tried, search_ok = [], set(), False

    for q in [f"{name} {loc} official website", f"{name} website"][:SEARCH_QUERIES]:
        results, ok = web_search(q)
        search_ok = search_ok or ok
        cands = []
        for rank, r in enumerate(results):
            url = r.get("href") or r.get("url") or ""
            host = urlparse(url).netloc.lower().split(":")[0]
            if not host:
                continue
            title, body = r.get("title", "") or "", r.get("body", "") or ""
            snippets.append(f"{title}. {body}")
            if is_directory(host):
                continue
            dom = host if is_free_host(host) else reg_domain(host)
            if dom in tried:
                continue
            strength = domain_strength(site_label(host), sig, joined)
            snip_cov = coverage(sig, (title + " " + body).lower())
            if strength >= 1 or (snip_cov >= 0.99 and len(sig) >= 2):
                cands.append((0 if strength else 1, rank, url, host, dom, strength))
        cands.sort(key=lambda c: (c[0], c[1]))
        for _, _, url, host, dom, strength in cands[:MAX_CANDIDATES]:
            tried.add(dom)
            found = evaluate_candidate(url, host, strength, sig, loc_words)
            if found:
                return found, snippets, True
    return None, snippets, search_ok

def make_row(name, address, found, snippets, search_ok, method):
    out = {"Company Name": name, "Address": address, "Method": method}
    if found:
        label, score, notes, f = assess_site(found["page"], is_free_host(found["host"]))
        texts = snippets + [f.get("title", ""), f.get("desc", ""), f.get("heads", ""), f.get("text", "")[:3000]]
        out.update({"Does it have website": "Yes", "Condition": label,
                    "Type of company": classify(name, texts, found["page"]["html"].lower()),
                    "Website URL": found["url"], "Quality score": score,
                    "Match confidence": found["conf"], "Notes": "; ".join(notes)})
    elif method == "Not searched":
        out.update({"Does it have website": "Not found (search skipped)", "Condition": "N/A",
                    "Type of company": classify(name, []), "Website URL": "", "Quality score": None,
                    "Match confidence": "", "Notes": "No site at the obvious domain names; web search was skipped"})
    elif search_ok:
        out.update({"Does it have website": "No", "Condition": "N/A",
                    "Type of company": classify(name, snippets), "Website URL": "", "Quality score": None,
                    "Match confidence": "",
                    "Notes": "No own website found (only directories / social pages / nothing)"})
    else:
        out.update({"Does it have website": "Unknown", "Condition": "N/A",
                    "Type of company": classify(name, []), "Website URL": "", "Quality score": None,
                    "Match confidence": "",
                    "Notes": "Web search failed (rate-limited?) - run the last cell again to retry"})
    return out

### Run website checker
This stage uses only domain guessing. Companies without a verified own-domain site are marked `No / Not found by domain guess`.

In [11]:
import os, pickle, hashlib, time, threading
from collections import deque
from tqdm.auto import tqdm

CHECKPOINT_FILE = f"{OUTPUT_DIR}/website_check_checkpoint.pkl"
RESULTS = {}

_input_signature = hashlib.sha256(
    (df.to_csv(index=False) + str(COMPANY_COL) + str(ADDRESS_COL)).encode()
).hexdigest()

if os.path.exists(CHECKPOINT_FILE):
    try:
        with open(CHECKPOINT_FILE, "rb") as f:
            saved = pickle.load(f)
        if saved.get("signature") == _INPUT_SIGNATURE:
            RESULTS.update(saved.get("results", {}))
            print(f"Resuming: {len(RESULTS)} companies already processed.")
    except Exception as e:
        print("Could not load checkpoint:", e)

def save_checkpoint():
    temp = CHECKPOINT_FILE + ".tmp"
    with open(temp, "wb") as f:
        pickle.dump({"signature": _INPUT_SIGNATURE, "results": RESULTS}, f,
                    protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(temp, CHECKPOINT_FILE)

MAIN_COLS = ["Company Name", "Does it have website", "Condition", "Company type", "Company type keywords"]
DETAIL_COLS = ["Company Name", "Address", "Does it have website", "Condition", "Company type",
               "Company type keywords", "Website URL", "Quality score", "Match confidence",
               "Method", "Notes"]

def build_output():
    rows = []
    for _, row in df.iterrows():
        name = str(row[COMPANY_COL]).strip()
        address = str(row[ADDRESS_COL]).strip() if pd.notna(row[ADDRESS_COL]) else ""
        result = RESULTS.get((name, address))
        if result:
            rows.append(result)
    detail = pd.DataFrame(rows, columns=DETAIL_COLS)
    return detail[MAIN_COLS], detail

def save_excel(path):
    main, detail = build_output()
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        main.to_excel(writer, sheet_name="Result", index=False)
        detail.to_excel(writer, sheet_name="Details", index=False)
    return main

def _run_task(func, item, box):
    try:
        box["value"] = func(item)
    except BaseException as exc:
        box["error"] = type(exc).__name__ + ": " + str(exc)

def run_parallel(func, items, workers, description, on_result):
    pending = deque(items)
    active = {}
    completed = 0
    bar = tqdm(total=len(items), desc=description)

    try:
        while pending or active:
            while pending and len(active) < workers:
                item = pending.popleft()
                box = {}
                thread = threading.Thread(target=_run_task, args=(func, item, box), daemon=True)
                thread.start()
                active[item] = (thread, box, time.monotonic())

            for item, (thread, box, started) in list(active.items()):
                expired = time.monotonic() - started >= STEP_A_LIMIT
                if thread.is_alive() and not expired:
                    continue

                found = None
                if not thread.is_alive():
                    if "value" in box:
                        found = box["value"]
                    elif "error" in box:
                        tqdm.write(f"Error for {item[0]}: {box['error']}")
                elif expired:
                    tqdm.write(f"Time limit ({STEP_A_LIMIT}s): {item[0]} — marked not found")

                del active[item]
                on_result(item, found)
                completed += 1
                bar.update(1)

                if completed % CHECKPOINT_EVERY == 0:
                    save_checkpoint()
                if completed % SAVE_EVERY == 0:
                    save_excel(f"{OUTPUT_DIR}/website_check_partial.xlsx")

            if active:
                time.sleep(0.05)
    except KeyboardInterrupt:
        print("Stopped. Completed work is checkpointed. Rerun this cell to resume.")
        raise
    finally:
        bar.close()
        save_checkpoint()

def type_keywords(name, texts):
    name_norm = re.sub(r"[^a-z0-9]+", " ", str(name).lower()).strip()
    name_joined = name_norm.replace(" ", "")
    blob = " ".join(texts).lower()
    matched = []

    for category, keywords in CATEGORIES.items():
        for keyword in keywords:
            k = keyword.lower()
            kj = k.replace(" ", "")
            in_name = re.search(r"\b" + re.escape(k) + r"\b", name_norm) or (
                len(kj) >= 4 and kj not in SUBSTRING_STOP and kj in name_joined
            )
            in_text = bool(re.search(r"\b" + re.escape(k) + (r"\b" if len(k) <= 4 else ""), blob))
            if in_name or in_text:
                matched.append(keyword)
            if len(matched) >= 4:
                return ", ".join(dict.fromkeys(matched))

    return ", ".join(dict.fromkeys(matched)) if matched else "Other / Unclassified"

def _guess(item):
    try:
        return guess_website(*item)
    except Exception as e:
        tqdm.write(f"Website-check error for {item[0]}: {type(e).__name__}: {e}")
        return None

def _on_guess(item, found):
    name, address = item

    if found:
        page = found["page"]
        label, score, notes, features = assess_site(page, is_free_host(found["host"]))
        texts = [features.get("title",""), features.get("desc",""),
                 features.get("heads",""), features.get("text","")[:3000]]
        company_type = classify(name, texts, page["html"].lower())
        keywords = type_keywords(name, texts)
        RESULTS[item] = {
            "Company Name": name, "Address": address,
            "Does it have website": "Yes", "Condition": label,
            "Company type": company_type, "Company type keywords": keywords,
            "Website URL": found["url"], "Quality score": score,
            "Match confidence": found["conf"], "Method": "Domain guess",
            "Notes": "; ".join(notes)
        }
    else:
        company_type = classify(name, [])
        keywords = type_keywords(name, [])
        RESULTS[item] = {
            "Company Name": name, "Address": address,
            "Does it have website": "No / Not found by domain guess",
            "Condition": "other", "Company type": company_type,
            "Company type keywords": keywords, "Website URL": "",
            "Quality score": None, "Match confidence": "",
            "Method": "Domain guess only",
            "Notes": "No matching own-domain website found; web search was not used."
        }

todo = []
for _, row in df.iterrows():
    name = str(row[COMPANY_COL]).strip()
    address = str(row[ADDRESS_COL]).strip() if pd.notna(row[ADDRESS_COL]) else ""
    if name:
        todo.append((name, address))
todo = list(dict.fromkeys(todo))

run_parallel(_guess, todo, WORKERS, "Website checking", _on_guess)

main = save_excel(WEBSITE_FILE)
save_checkpoint()
print(f"Saved website results: {WEBSITE_FILE}")
print(main["Does it have website"].value_counts())
main.head()


Website checking:   0%|          | 0/4953 [00:00<?, ?it/s]

NameError: name '_INPUT_SIGNATURE' is not defined

## 4. Lead qualification
Classifies each company using its website presence, condition, company type, and keywords. It preserves the exact input row count and does not merge on company name.

In [ ]:
# Use the Result sheet from website checking.
website_df = pd.read_excel(WEBSITE_FILE, sheet_name="Result", dtype=str)
website_df = website_df.rename(columns={"Type of company": "Company type"})

# Preserve company contact details from the cleaned data.
contact_cols = [COMPANY_COL, EMAIL_COL, ADDRESS_COL, STATE_COL]
if "City Name" in df.columns:
    contact_cols.append("City Name")
contact_df = df[contact_cols].copy()
contact_df = contact_df.rename(columns={COMPANY_COL: "Company Name", ADDRESS_COL: "Address"})

# Merge website results positionally: website Result sheet was built in cleaned-data order.
for col in ["Does it have website", "Condition", "Company type", "Company type keywords"]:
    if col not in website_df.columns:
        website_df[col] = "other"

lead_df = contact_df.copy()
for col in ["Does it have website", "Condition", "Company type", "Company type keywords"]:
    lead_df[col] = website_df[col].values[:len(lead_df)]

for col in ["Company Name", "Does it have website", "Condition", "Company type", "Company type keywords"]:
    lead_df[col] = lead_df[col].fillna("other").astype(str).str.strip().replace(
        {"": "other", "nan": "other", "NaN": "other"}
    )

CUSTOM_BUILD_INDUSTRIES = {
    "E-commerce / Online Retail": 3, "IT & Software": 3, "Education & Training": 2,
    "Hospitality & Travel": 2, "Financial Services": 2, "Media, Printing & Packaging": 2,
    "Pharma & Healthcare": 1, "Retail & Consumer Goods": 1,
    "Construction & Real Estate": 1, "Manufacturing & Engineering": 1,
    "Other / Unclassified": 0
}

PULSEBOARD_INDUSTRIES = {
    "Construction & Real Estate": 3, "Manufacturing & Engineering": 3,
    "Logistics & Transport": 3, "Trading & Distribution": 2, "Education & Training": 2,
    "Pharma & Healthcare": 2, "Media, Printing & Packaging": 2, "Hospitality & Travel": 2,
    "Financial Services": 2, "Retail & Consumer Goods": 1,
    "E-commerce / Online Retail": 1, "Other / Unclassified": 0
}

CUSTOM_KEYWORDS = {
    "ecommerce":3,"e-commerce":3,"online shop":3,"online store":3,"website":3,"web":2,
    "app":3,"digital":2,"software":3,"technology":2,"technologies":2,"learning":2,
    "media":2,"fashion":1,"jewellers":1,"retail":1,"properties":2,"real estate":2,
    "booking":3,"payment":3,"data":1
}

PULSEBOARD_KEYWORDS = {
    "construction":3,"contractors":3,"projects":3,"project":3,"engineering":2,
    "manufacturing":2,"trading":2,"traders":2,"distributor":2,"distribution":2,
    "supply":2,"logistics":3,"transport":2,"garage":1,"power":1,"steel":1,"fabric":1,
    "plastic":1,"led":1,"medical":1,"team":2,"operations":3
}

def split_keywords(value):
    if not isinstance(value, str):
        return []
    return [k.strip().lower() for k in value.split(",")
            if k.strip() and k.strip().lower() != "other"]

def score_company(row):
    company_type = str(row["Company type"]).strip()
    keywords = split_keywords(row["Company type keywords"])
    condition = str(row["Condition"]).strip().lower()
    has_website = str(row["Does it have website"]).strip().lower() == "yes"

    custom_score = CUSTOM_BUILD_INDUSTRIES.get(company_type, 0)
    pulse_score = PULSEBOARD_INDUSTRIES.get(company_type, 0)
    custom_hits, pulse_hits = [], []

    for keyword in keywords:
        if keyword in CUSTOM_KEYWORDS:
            custom_score += CUSTOM_KEYWORDS[keyword]
            custom_hits.append(keyword)
        if keyword in PULSEBOARD_KEYWORDS:
            pulse_score += PULSEBOARD_KEYWORDS[keyword]
            pulse_hits.append(keyword)

    if condition == "requires work":
        custom_score += 2
    elif condition == "very poor":
        custom_score += 3
    elif condition == "new / modern":
        pulse_score += 1

    if has_website:
        custom_score += 1
        pulse_score += 1

    return custom_score, pulse_score, custom_hits, pulse_hits

def classify_company(row):
    custom_score, pulse_score, custom_hits, pulse_hits = score_company(row)

    if custom_score >= 5 and custom_score > pulse_score:
        category = "Custom Build Candidate"
    elif pulse_score >= 5 and pulse_score > custom_score:
        category = "PulseBoard Candidate"
    elif custom_score >= 4 and pulse_score >= 4:
        category = "Custom Build Candidate" if custom_score >= pulse_score else "PulseBoard Candidate"
    elif custom_score >= 4:
        category = "Custom Build Candidate"
    elif pulse_score >= 4:
        category = "PulseBoard Candidate"
    else:
        category = "Low Fit / Unqualified"

    if category == "Custom Build Candidate":
        criticality = "High" if custom_score >= 8 else "Medium" if custom_score >= 6 else "Low"
    elif category == "PulseBoard Candidate":
        criticality = "High" if pulse_score >= 8 else "Medium" if pulse_score >= 6 else "Low"
    else:
        criticality = "Low"

    return category, criticality, custom_hits, pulse_hits, custom_score, pulse_score

def make_justification(row, category, custom_hits, pulse_hits, custom_score, pulse_score):
    name = row["Company Name"]
    company_type = row["Company type"] if row["Company type"].lower() != "other" else "its industry"
    keywords = split_keywords(row["Company type keywords"])
    keyword_text = ", ".join(keywords[:4]) if keywords else "no specific keywords"

    if category == "Custom Build Candidate":
        return (f"{name} operates in {company_type} and shows custom digital-platform potential "
                f"(custom score {custom_score}; signals: {', '.join(custom_hits[:4]) or 'industry category'}). "
                f"It may benefit from a custom website, app, catalogue, booking flow, or customer-facing platform. "
                f"Missing fields were handled using company name and available keywords.")
    if category == "PulseBoard Candidate":
        return (f"{name} operates in {company_type} and shows execution or collaboration potential "
                f"(PulseBoard score {pulse_score}; signals: {', '.join(pulse_hits[:4]) or 'industry category'}). "
                f"PulseBoard can support task tracking, project updates, team communication, and operational follow-ups. "
                f"Missing fields were handled using company name and available keywords.")
    return (f"{name} has insufficient product-fit signals based on its category and keywords "
            f"(custom score {custom_score}, PulseBoard score {pulse_score}). "
            f"Missing fields were treated as 'other' and inferred only from available context.")

qualification_rows = []
for _, row in lead_df.iterrows():
    category, criticality, custom_hits, pulse_hits, custom_score, pulse_score = classify_company(row)
    qualification_rows.append({
        "Assigned Product Category": category,
        "Criticality": criticality,
        "Justification / Rationale": make_justification(
            row, category, custom_hits, pulse_hits, custom_score, pulse_score
        ),
        "Custom Build Score": custom_score,
        "PulseBoard Score": pulse_score,
        "Custom Build Signals": ", ".join(custom_hits) if custom_hits else "other",
        "PulseBoard Signals": ", ".join(pulse_hits) if pulse_hits else "other"
    })

qualification_df = pd.DataFrame(qualification_rows)
for col in qualification_df.columns:
    lead_df[col] = qualification_df[col].values

final_columns = [
    "Company Name", "Does it have website", "Condition", "Company type",
    "Company type keywords", "Assigned Product Category", "Criticality",
    "Justification / Rationale", "Custom Build Score", "PulseBoard Score",
    "Custom Build Signals", "PulseBoard Signals"
]
if EMAIL_COL in lead_df.columns: final_columns.insert(1, EMAIL_COL)
if ADDRESS_COL in lead_df.columns and "Address" not in final_columns:
    final_columns.insert(1, "Address")
if STATE_COL in lead_df.columns: final_columns.insert(2, STATE_COL)
if "City Name" in lead_df.columns: final_columns.insert(3, "City Name")

final_df = lead_df[[c for c in final_columns if c in lead_df.columns]]

assert len(final_df) == len(df), "Row count changed during qualification."

with pd.ExcelWriter(QUALIFIED_FILE, engine="openpyxl") as writer:
    final_df.to_excel(writer, sheet_name="Lead Qualification", index=False)
    ws = writer.sheets["Lead Qualification"]
    for i, col in enumerate(final_df.columns, 1):
        width = min(60, max([len(str(col))] + final_df[col].astype(str).head(200).map(len).tolist()) + 2)
        ws.column_dimensions[ws.cell(row=1, column=i).column_letter].width = width

print(f"Saved qualified leads: {QUALIFIED_FILE}")
print("\nAssigned categories:")
print(final_df["Assigned Product Category"].value_counts())
print("\nCriticality:")
print(final_df["Criticality"].value_counts())
final_df.head(20)


## 5. Download outputs
If running in Google Colab, this downloads all three output files. On Windows, the files are already saved in your configured `OUTPUT_DIR`.

In [ ]:
try:
    from google.colab import files
    for path in [CLEANED_FILE, WEBSITE_FILE, QUALIFIED_FILE]:
        files.download(path)
except ImportError:
    print("Outputs saved locally:")
    for path in [CLEANED_FILE, WEBSITE_FILE, QUALIFIED_FILE]:
        print(path)
